# 01 — GloFAS Mini-Pipeline: Download → Process → ML-Ready

This notebook does **three things**:
1. **Download** a short GloFAS window (3 variables) from the Copernicus EWDS API — *cache-first*,
   so it never re-downloads.
2. **Process** — unpack, merge the variables onto one grid, fix coordinates, quality-control.
3. **Export ML-ready artifacts** — a normalized `(time, channel, lat, lon)` cube, the
   normalization statistics, and a small metadata manifest.


## 0. Requirements & one-time API setup

```bash
pip install "cdsapi>=0.7.7" xarray netcdf4 h5netcdf numpy pandas
```

**Credentials (only needed to download new data):**
1. Register at <https://ewds.climate.copernicus.eu> and copy your Personal Access Token.
2. Accept the *CEMS-FLOODS licence* once on the
   [dataset page](https://ewds.climate.copernicus.eu/datasets/cems-glofas-historical?tab=download).
3. Create `~/.cdsapirc`:
   ```
   url: https://ewds.climate.copernicus.eu/api
   key: <your-token>
   ```

If you already have a processed file cached, you can run everything except the download cell
without credentials.


In [22]:
# ============================================================
# CELL 1 — Configuration (edit this cell, then Run All)
# ============================================================
from pathlib import Path

# --- The small window to experiment with -------------------------------------
# Default = June 2022 (the Sylhet flood month). For a 1-WEEK test, keep one month
# here and set DAY_RANGE below; for a different month just change these.
YEAR   = "2022"
MONTHS = ["03","04","05","06","07"]                 # one or more months, e.g. ["06"] or ["05","06"]
DAY_RANGE = range(1, 31)        # 1..30 = whole month; e.g. range(14, 22) = one week

# --- Region: Bangladesh + NE India box [N, W, S, E] --------------------------
# Includes Meghalaya/Assam (upstream source of NE Bangladesh floods).
AREA = [27.5, 87.0, 21.0, 94.0]
REGION_NAME = "bd_ne_india"

# --- GloFAS product settings (v4.0 = 0.05 deg, current) ----------------------
GLOFAS = {
    "dataset": "cems-glofas-historical",
    "system_version": ["version_4_0"],
    "hydrological_model": ["lisflood"],
    "product_type": ["consolidated"],
    "variables": [
        "river_discharge_in_the_last_24_hours",   # dis24  [m3/s]
        "runoff_water_equivalent",                # runoff [kg/m2 == mm/day]
        "soil_wetness_index",                     # SWI    [0..1]
    ],
}

# --- Folders (kept out of git via .gitignore) --------------------------------
RAW_DIR  = Path("data_raw");   RAW_DIR.mkdir(exist_ok=True)    # downloaded zips/nc
PROC_DIR = Path("data_ready"); PROC_DIR.mkdir(exist_ok=True)   # ML-ready outputs

DAYS = [f"{d:02d}" for d in DAY_RANGE]
TAG  = f"{REGION_NAME}_{YEAR}-{MONTHS[0]}_to_{YEAR}-{MONTHS[-1]}_d{DAYS[0]}-{DAYS[-1]}"
print(f"window tag : {TAG}")
print(f"variables  : {len(GLOFAS['variables'])} | days: {DAYS[0]}..{DAYS[-1]} | months: {MONTHS}")


window tag : bd_ne_india_2022-03_to_2022-07_d01-30
variables  : 3 | days: 01..30 | months: ['03', '04', '05', '06', '07']


## Step 1 — Download (cache-first)

The cell submits **one request** for all three variables over the window and saves the resulting
zip in `data_raw/`. If the zip already exists, it is reused — the API is never contacted again.
The EWDS queues requests server-side; a one-month, one-region pull is a few MB and usually returns
in 1–5 minutes.


In [23]:
# ============================================================
# CELL 2 — Download the window (skips if already present)
# ============================================================
zip_path = RAW_DIR / f"glofas_{TAG}.zip"

def download():
    if zip_path.exists():
        print(f"cache hit -> {zip_path.name} (delete it to force re-download)")
        return True
    try:
        import cdsapi
    except ImportError:
        print("cdsapi not installed -> cannot download. `pip install cdsapi`.")
        return False
    request = {
        "system_version": GLOFAS["system_version"],
        "hydrological_model": GLOFAS["hydrological_model"],
        "product_type": GLOFAS["product_type"],
        "variable": GLOFAS["variables"],
        "hyear": [YEAR], "hmonth": MONTHS, "hday": DAYS,
        "data_format": "netcdf", "download_format": "zip", "area": AREA,
    }
    try:
        client = cdsapi.Client()
        print("submitting EWDS request (queued server-side, ~1-5 min)...")
        client.retrieve(GLOFAS["dataset"], request).download(str(zip_path))
        print(f"downloaded -> {zip_path.name} ({zip_path.stat().st_size/1e6:.1f} MB)")
        return True
    except Exception as exc:
        print(f"download failed: {str(exc)[:250]}")
        print("Check ~/.cdsapirc and that you accepted the dataset licence.")
        return False

have_zip = download()


cache hit -> glofas_bd_ne_india_2022-03_to_2022-07_d01-30.zip (delete it to force re-download)


## Step 2 — Process: unpack, merge, fix coordinates

GloFAS returns **one NetCDF per variable** inside the zip. We:
- unpack each into `data_raw/`,
- **merge** them into a single dataset (they share the same grid and dates),
- **normalize coordinates** — rename `valid_time→time`, ensure latitude is descending (GloFAS
  convention), sort by time,
- **identify variables by units** (`m3/s`→discharge, `kg/m2`→runoff, dimensionless→SWI) so the
  code is robust to GloFAS's short variable names (`dis24`, `rowe`, `swir`).


In [24]:
# ============================================================
# CELL 3 — Unpack + merge + normalize
# ============================================================
import zipfile
import numpy as np
import xarray as xr

def open_nc(path):
    for eng in ("h5netcdf", "netcdf4", None):
        try:
            return xr.open_dataset(path, engine=eng) if eng else xr.open_dataset(path)
        except (ValueError, OSError):
            continue
    raise IOError(f"could not open {path}")

def normalize(ds):
    ren = {a: b for a, b in [("valid_time", "time"), ("lat", "latitude"), ("lon", "longitude")]
           if a in ds.dims or a in ds.coords}
    ds = ds.rename(ren)
    if ds.latitude.values[0] < ds.latitude.values[-1]:
        ds = ds.sortby("latitude", ascending=False)
    return ds.sortby("time")

assert zip_path.exists(), "No downloaded zip found — run Cell 2 with credentials first."
parts = []
with zipfile.ZipFile(zip_path) as z:
    for n in [n for n in z.namelist() if n.endswith(".nc")]:
        p = RAW_DIR / f"_part_{Path(n).name}"
        p.write_bytes(z.read(n)); parts.append(p)
ds = normalize(xr.merge([open_nc(p) for p in parts]))
for p in parts:
    p.unlink()

# Map friendly names by units
CH = {}
for v in ds.data_vars:
    u = str(ds[v].attrs.get("units", "")).replace("**", "").replace("^", "").lower()
    if "m3" in u:      CH["discharge"] = v
    elif "kg" in u:    CH["runoff"] = v
    else:              CH["soil_wetness"] = v
print("variables found:", CH)
print("dims:", dict(ds.sizes), "| time:",
      str(ds.time.values[0])[:10], "->", str(ds.time.values[-1])[:10])


variables found: {'runoff': 'rowe', 'soil_wetness': 'swir', 'discharge': 'dis24'}
dims: {'time': 150, 'latitude': 130, 'longitude': 140} | time: 2022-03-02 -> 2022-07-31


## Step 3 — Quality control

Before ML, check the data is sane. We report per-variable **range, missing fraction, and fill
values**, then replace any sentinel/fill values with `NaN`. GloFAS marks non-river / masked cells
with very large or negative fill values; runoff and SWI are dense (defined everywhere), discharge
is effectively a river network (most land cells ~0).


In [25]:
# ============================================================
# CELL 4 — Quality control report + fill-value cleaning
# ============================================================
import pandas as pd

rows = []
for name, v in CH.items():
    arr = ds[v]
    fill = arr.attrs.get("_FillValue", arr.attrs.get("missing_value", None))
    if fill is not None:
        arr = arr.where(arr != fill)
    # also guard against absurd sentinels
    arr = arr.where(np.abs(arr) < 1e19)
    ds[v] = arr
    vals = arr.values
    rows.append({
        "channel": name, "var": v, "units": arr.attrs.get("units", "?"),
        "min": float(np.nanmin(vals)), "max": float(np.nanmax(vals)),
        "mean": float(np.nanmean(vals)),
        "missing_%": round(100 * np.isnan(vals).mean(), 2),
    })
qc = pd.DataFrame(rows)
print(qc.to_string(index=False))
assert (qc["missing_%"] < 90).all(), "A channel is almost entirely missing — check the download."
print("\nQC passed.")


     channel   var      units      min           max       mean  missing_%
      runoff  rowe   kg m**-2 0.000000    222.753845   3.943750       9.25
soil_wetness  swir    Numeric 0.205274      0.986368   0.727551       9.25
   discharge dis24 m**3 s**-1 0.000000 104540.281250 386.098572       9.25

QC passed.


## Step 4 — Build the ML-ready cube

Now assemble the model input. Steps:

1. **Stack** the three variables into one array with a `channel` dimension →
   shape `(time, channel, lat, lon)`.
2. **Fill remaining gaps** — discharge's off-river `NaN`s become 0 (no flow); runoff/SWI gaps
   (rare) are filled with the channel mean.
3. **Standardize** each channel to mean 0 / std 1 and **save the stats** (so predictions can be
   converted back to physical units later).
4. **Save** three artifacts to `data_ready/`:
   - `*_raw.nc` — physical-unit cube (for plotting / inspection),
   - `*_norm.nc` — standardized cube (the actual ML input),
   - `*_stats.json` + `*_manifest.json` — normalization stats and metadata.


In [26]:
# ============================================================
# CELL 5 — Assemble, standardize, save
# ============================================================
import json

order = [k for k in ["discharge", "runoff", "soil_wetness"] if k in CH]
raw = xr.concat([ds[CH[k]] for k in order], dim="channel")
raw = raw.assign_coords(channel=order).rename("glofas").transpose("time", "channel", "latitude", "longitude")

# Fill gaps: discharge off-river -> 0; others -> channel mean
filled = raw.copy()
for k in order:
    sl = filled.sel(channel=k)
    fill_val = 0.0 if k == "discharge" else float(sl.mean(skipna=True))
    filled.loc[dict(channel=k)] = sl.fillna(fill_val)

# Standardize per channel (mean 0 / std 1), keep stats
stats = {}
norm = filled.copy()
for k in order:
    sl = filled.sel(channel=k)
    mu, sd = float(sl.mean()), float(sl.std())
    sd = sd if sd > 1e-8 else 1.0
    norm.loc[dict(channel=k)] = (sl - mu) / sd
    stats[k] = {"mean": mu, "std": sd}

# Save
raw_path  = PROC_DIR / f"glofas_{TAG}_raw.nc"
norm_path = PROC_DIR / f"glofas_{TAG}_norm.nc"
filled.to_dataset(name="glofas").to_netcdf(raw_path, engine="h5netcdf")
norm.to_dataset(name="glofas").to_netcdf(norm_path, engine="h5netcdf")

manifest = {
    "tag": TAG, "region": REGION_NAME, "area_NWSE": AREA,
    "year": YEAR, "months": MONTHS, "days": [DAYS[0], DAYS[-1]],
    "channels": order,
    "shape_time_channel_lat_lon": list(norm.shape),
    "resolution_deg": 0.05, "time_step": "daily",
    "source": "GloFAS-ERA5 v4.0 (cems-glofas-historical, EWDS)",
    "citation": "Harrigan et al. (2020) ESSD; DOI 10.24381/cds.a4fdd6b9",
    "normalization": "per-channel standardize (mean 0 / std 1); see *_stats.json",
    "files": {"raw": raw_path.name, "norm": norm_path.name},
}
json.dump(stats, open(PROC_DIR / f"glofas_{TAG}_stats.json", "w"), indent=2)
json.dump(manifest, open(PROC_DIR / f"glofas_{TAG}_manifest.json", "w"), indent=2)

print("ML-ready cube shape (time, channel, lat, lon):", norm.shape)
print("channels:", order)
for f in sorted(PROC_DIR.glob(f"glofas_{TAG}_*")):
    print(f"  wrote {f.name}  ({f.stat().st_size/1e3:.0f} kB)")


ML-ready cube shape (time, channel, lat, lon): (150, 3, 130, 140)
channels: ['discharge', 'runoff', 'soil_wetness']
  wrote glofas_bd_ne_india_2022-03_to_2022-07_d01-30_manifest.json  (1 kB)
  wrote glofas_bd_ne_india_2022-03_to_2022-07_d01-30_norm.nc  (32780 kB)
  wrote glofas_bd_ne_india_2022-03_to_2022-07_d01-30_raw.nc  (32780 kB)
  wrote glofas_bd_ne_india_2022-03_to_2022-07_d01-30_stats.json  (0 kB)


## Done — what you have now

In `data_ready/` (per window):

| File | What it is | Use |
|---|---|---|
| `*_norm.nc` | standardized `(time, channel, lat, lon)` cube | **the ML input** |
| `*_raw.nc` | same cube in physical units | plotting / inspection |
| `*_stats.json` | per-channel mean & std | invert predictions back to m³/s, mm, etc. |
| `*_manifest.json` | window/region/shape/source metadata | provenance for the experiment |

**Next:** open **`02_visualize.ipynb`** to sanity-check the cube (maps, time series, distributions)
before any modeling. To experiment with a different window, edit **Cell 1** and Run All — the
download is cached, processing is deterministic.
